**PREPARATION AND ANALYSIS OF SIESTA SIMULATIONS WITH PYTHON**
--------------------------------------------------------------

Using the pythin libray sisl, we can prepare simulations for SIESTA and analyze results.
This notebook is based on the tutorials available here:

https://sisl.readthedocs.io/en/latest/tutorials/index.html#siesta-transiesta-support


Setup
-----
We first import all required libraries

In [ ]:
from sisl import *
import numpy as np
import matplotlib.pyplot as plt

Visualization of a structure
----------------------------
Example of loading and visualization of a fdf file with coordinates. 

In [ ]:
system = Geometry.read("./STRUCT.fdf")

In [ ]:
system.plot()

In [ ]:
system.plot(axes="xy")

## Analysis of SIESTA Results
**Load SIESTA Calculation**

Once the SIESTA run has finished, to load SIESTA results into python you can use the following sisl command:

```python
get_sile("path/to/my/file.fdf").read_hamiltonian()
```

This will read the Hamiltonian with all the information of the system corresponding to this Hamiltonian

In [ ]:
fdf = get_sile("RUN.fdf")

In [ ]:
H = fdf.read_hamiltonian()

In [ ]:
print(H)

**Show Data employed in the calculation**

Before analysing the diagonalisation of H, we can show the geometry of the system and the basis set employed

In [ ]:
system = H.geometry

In [ ]:
system.plot(axes="xy")

In [ ]:
# FUNCTION PLOT ORBITALS
def plot_atom(atom):
    no = len(atom) # number of orbitals
    nx = no // 4
    ny = no // nx
    if nx * ny < no:
        nx += 1
    fig, axs = plt.subplots(nx, ny, figsize=(20, 5*nx))
    fig.suptitle('Atom: {}'.format(atom.symbol), fontsize=14)
    def my_plot(i, orb):
            grid = orb.toGrid(atom=atom)
            # Also write to a cube file
            grid.write('{}_{}.cube'.format(atom.symbol, orb.name()))
            c, r = i // 4, (i - 4) % 4
            if nx == 1:
                ax = axs[r]
            else:
                ax = axs[c][r]
            ax.imshow(grid.grid[:, :, grid.shape[2] // 2])
            ax.set_title(r'${}$'.format(orb.name(True)))
            ax.set_xlabel(r'$x$ [Ang]')
            ax.set_ylabel(r'$y$ [Ang]')
    i = 0
    for orb in atom:
            my_plot(i, orb)
            i += 1
    if i < nx * ny:
            # This removes the empty plots
            for j in range(i, nx * ny):
                c, r = j // 4, (j - 4) % 4
                if nx == 1:
                    ax = axs[r]
                else:
                    ax = axs[c][r]
                fig.delaxes(ax)
            plt.draw()

In [ ]:
plot_atom(system.atoms[0])
plot_atom(system.atoms[1])
plt.show()

## Eigenstates

Once you have a hamiltonian, you can get all Eigenstates with `H.eigenstate()`. Then you can loop this object to get each individual eigenstate. 
- Each eigenstate has its energy stored under the `.eig` property.
- We can find the HOMO and LUMO.
- We can use the `.dos`  method to obtain the density of states DOS(E).
- In sisl we can plot pdos directly in a given Energy range with something like `H.plot.pdos(data_Erange=[-10,10],nE=1200,Erange=[-10,10]) `

In [ ]:
es=H.eigenstate()

In [ ]:
es.eig

In [ ]:
idx_lumo = (es.eig > 0).nonzero()[0][0] #trick to obtain the LUMO as first positive eigenvalue
idx_homo = idx_lumo-1

In [ ]:
print(idx_lumo)

In [ ]:
print("HOMO E=",es.eig[idx_homo]," eV")
print("LUMO E=",es.eig[idx_lumo]," eV")

-----------------------------------------------------------------------------------
## Exporting Wavefunctions for 3D plotting

Create a grid and fill the grid with the probability density. The result can be sved in .cube format:

https://paulbourke.net/dataformats/cube/


**To compute wavefunctions on the grid:**

To compute $\psi (\vec{r})$ you need three things.

1. **The eigenstate coefficients.** Once you have a hamiltonian, you can get all of them with `H.eigenstate()`. Then you can loop this object to get each individual eigenstate. Each eigenstate has its energy stored under the `.eig` property.
2. **A grid of points in space.** You can create one with `sisl.Grid(geometry, shape=(100, 100, 100))`. This will create a grid of $100x100x100$ points within the cell of your geometry.
3. **A function to project the wavefunction into the grid.** The eigenstate object has a `wavefunction` method ([docs](https://zerothi.github.io/sisl/api/generated/sisl.physics.electron.EigenstateElectron.html#sisl.physics.electron.EigenstateElectron.wavefunction)) that will project the wavefunction into an already initialized grid.
4. Once you have the wavefunction, you can calculate the electron density $\psi_i(\vec{r}) \psi_i^*(\vec{r})$.
5. The result can be saved in a '.cube' file and plotted with VMD 

This can always be done as in our first SIESTA lab, with a water molecule and with $C_{60}$.


In [ ]:
system.lattice.origin = [-4, -4, -4]
g = Grid(0.2, lattice=system.lattice)  #defines a grid matching with our system

Save desired states (for example, HOMO and LUMO) in cube file

In [ ]:
es[idx_homo].wavefunction(g)
g.write('HOMO.cube')
print('Real space integrated wavefunction: {:.4f}'.format((np.absolute(g.grid) ** 2).sum() * g.dvolume))
g.fill(0) # reset the grid values to 0

In [ ]:
es[idx_lumo].wavefunction(g)
g.write('LUMO.cube')
print('Real space integrated wavefunction: {:.4f}'.format((np.absolute(g.grid) ** 2).sum() * g.dvolume))
g.fill(0) # reset the grid values to 0

Save Electron Density

In [ ]:
g2 = Grid(0.2, lattice=system.lattice) 

In [ ]:
es[idx_lumo].wavefunction(g)
g2=g*g.grid.conj() 
g2.write('LUMOdens.cube')
g2.fill(0) # reset the grid values to 0

We can easily save the electron density including a range of states

In [ ]:
Gt = Grid(0.2, lattice=system.lattice) 
for n in range(idx_lumo-4,idx_lumo):
    print("adding state",n)
    g.fill(0)
    es[n].wavefunction(g)
    Gt = Gt + g*g.grid.conj()
Gt.write('several_states.cube')
Gt.fill(0)

    